# Southern Indiana Housing and Public School Analysis 
This project analyzes housing sales data from multiple resources (Redfin and Realtor.com Data Centers) along with the results of the ILEARN standardized test for school corporations in Bartholomew, Clark, and Floyd counties located in Southern Indian. The goal of this project is to anticipate housing prices within the better school districts within those counties in anticipation of a potential move for my family.

- In this notebook I will read in the housing data and clean it. 

In [19]:
#Import necessary libraries and such for project. And some extra stuff, just in case.
import pandas as pd
import matplotlib
from pandas import DataFrame
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import datetime
import seaborn as sns
import dash 
from dash import dcc

Now I will read in data I sourced from Redfin's Data Center - it includes data for all of the United State

#### Housing Data by County for the state of Indiana

In [20]:
# Reading in the mother file with all housing data by county

housing_data= pd.read_csv('Data_Sources\county_market_tracker_clean.zip')
housing_data.head()

,period_begin,period_end,table_id,region,state,state_code,property_type,property_type_id,median_sale_price,median_list_price,median_ppsf,median_ppsf_yoy,median_list_ppsf,median_list_ppsf_yoy,inventory,inventory_yoy,sold_above_list,parent_metro_region,parent_metro_region_metro_code,last_updated
0,4/1/2015,4/30/2015,2550,"Bradley County, TN",Tennessee,TN,Condo/Co-op,3,199900.0,NaN,137.957212,NaN,NaN,NaN,3.0,NaN,0.000000,"Cleveland, TN",17420.0,2/10/2025 14:21
1,2/1/2014,2/28/2014,1958,"Dutchess County, NY",New York,NY,Townhouse,13,190250.0,269500.0,133.055161,-0.017109,127.048023,0.062799,21.0,-0.125000,0.000000,"Poughkeepsie, NY",39100.0,2/10/2025 14:21
2,6/1/2023,6/30/2023,1643,"Holt County, MO",Missouri,MO,Single Family Residential,6,75500.0,229500.0,40.810811,-0.724764,120.578231,0.525863,13.0,1.166667,0.000000,Missouri nonmetropolitan area,NaN,2/10/2025 14:21
3,3/1/2015,3/31/2015,1518,"Adams County, MS",Mississippi,MS,All Residential,-1,94900.0,160750.0,70.059310,-0.224828,81.662621,0.217752,57.0,-0.123077,0.181818,"Natchez, MS",35020.0,2/10/2025 14:21
4,2/1/2016,2/29/2016,806,"Washington County, IL",Illinois,IL,Single Family Residential,6,49000.0,76500.0,32.236842,-0.121009,67.222597,-0.260068,57.0,0.212766,0.400000,Illinois nonmetropolitan area,NaN,2/10/2025 14:21


I can see from looking at the first five rows that the dataset I found has housing data from across the United States going back at leas 11 years. That's more than I need, but I want to get a better idea of what the whole dataframe looks like. Let me see the shape and datatypes I'll be working with. 

In [21]:
# What are the columns? I thought I saw pricing per sqft. I need to list all the column names.
# Displaying the size and data types of our data
hdata_shape = housing_data.shape
print(f'The DataFrame has {hdata_shape[0]} rows and {hdata_shape[1]} columns! \n')
print(housing_data.info()) 

The DataFrame has 1048575 rows and 20 columns! 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 20 columns):
 #   Column                          Non-Null Count    Dtype  
---  ------                          --------------    -----  
 0   period_begin                    1048575 non-null  object 
 1   period_end                      1048575 non-null  object 
 2   table_id                        1048575 non-null  int64  
 3   region                          1048575 non-null  object 
 4   state                           1048575 non-null  object 
 5   state_code                      1048575 non-null  object 
 6   property_type                   1048575 non-null  object 
 7   property_type_id                1048575 non-null  int64  
 8   median_sale_price               1047819 non-null  float64
 9   median_list_price               963668 non-null   float64
 10  median_ppsf                     1037705 non-null  float64
 11  median_ppsf_yo

I'm not interested in year over year data because, for this project, I'm not interested in how the pricing is trending in those counties. I just want a recent snapshot. I'll also have to control for property type since we're only interested in purchasing a single family home. The columns I'll keep are: period_begin, period_end, region (this is the county in this data), state, property_type, median_sale_price, and median_ppsf. 

In [22]:
# drop columns now that I know which columns I want
housing_data_clean = housing_data[['period_begin', 'period_end', 'region', 'state','property_type', 'median_sale_price', 'median_list_price', 'median_ppsf']]
housing_data_clean


,period_begin,period_end,region,state,property_type,median_sale_price,median_list_price,median_ppsf
0,4/1/2015,4/30/2015,"Bradley County, TN",Tennessee,Condo/Co-op,199900.0,NaN,137.957212
1,2/1/2014,2/28/2014,"Dutchess County, NY",New York,Townhouse,190250.0,269500.0,133.055161
2,6/1/2023,6/30/2023,"Holt County, MO",Missouri,Single Family Residential,75500.0,229500.0,40.810811
3,3/1/2015,3/31/2015,"Adams County, MS",Mississippi,All Residential,94900.0,160750.0,70.059310
4,2/1/2016,2/29/2016,"Washington County, IL",Illinois,Single Family Residential,49000.0,76500.0,32.236842
...,...,...,...,...,...,...,...,...
1048570,2/1/2015,2/28/2015,"Apache County, AZ",Arizona,Single Family Residential,151000.0,NaN,91.972077
1048571,1/1/2021,1/31/2021,"Forsyth County, NC",North Carolina,All Residential,209000.0,225000.0,116.748840
1048572,6/1/2022,6/30/2022,"Boone County, WV",West Virginia,Condo/Co-op,35000.0,NaN,7.000000
1048573,4/1/2021,4/30/2021,"Livingston Parish, LA",Louisiana,All Residential,212000.0,219900.0,106.772422


Next I'll remove all the rows that do not have Indiana as a state by focusing on specific counties. I'll put these into three separate dataframes and then merge them.

In [23]:
#removes rows that do not contai
housing_data_bart = housing_data_clean.loc[housing_data_clean["region"] == "Bartholomew County, IN"]
housing_data_clark = housing_data_clean.loc[housing_data_clean["region"] == "Clark County, IN"]
housing_data_floyd = housing_data_clean.loc[housing_data_clean["region"]== "Floyd County, IN"]

display(housing_data_clark, housing_data_floyd, housing_data_bart)

,period_begin,period_end,region,state,property_type,median_sale_price,median_list_price,median_ppsf
643,1/1/2016,1/31/2016,"Clark County, IN",Indiana,Condo/Co-op,95950.0,NaN,77.715124
1796,12/1/2014,12/31/2014,"Clark County, IN",Indiana,Single Family Residential,129400.0,139950.0,80.013918
1842,12/1/2015,12/31/2015,"Clark County, IN",Indiana,Condo/Co-op,39000.0,139900.0,30.468750
4393,11/1/2016,11/30/2016,"Clark County, IN",Indiana,All Residential,137350.0,150000.0,79.753239
7871,12/1/2012,12/31/2012,"Clark County, IN",Indiana,All Residential,122000.0,120000.0,71.851226
...,...,...,...,...,...,...,...,...
1036652,11/1/2024,11/30/2024,"Clark County, IN",Indiana,Single Family Residential,260000.0,244900.0,171.232980
1040135,2/1/2021,2/28/2021,"Clark County, IN",Indiana,Condo/Co-op,246750.0,249900.0,154.681779
1040502,7/1/2024,7/31/2024,"Clark County, IN",Indiana,Condo/Co-op,169500.0,253245.0,150.223214
1046036,4/1/2018,4/30/2018,"Clark County, IN",Indiana,Single Family Residential,154000.0,169900.0,97.244733


,period_begin,period_end,region,state,property_type,median_sale_price,median_list_price,median_ppsf
24,1/1/2014,1/31/2014,"Floyd County, IN",Indiana,Single Family Residential,108500.0,139900.0,58.016878
2800,8/1/2017,8/31/2017,"Floyd County, IN",Indiana,All Residential,182000.0,164500.0,91.635338
6488,9/1/2017,9/30/2017,"Floyd County, IN",Indiana,All Residential,160253.0,149450.0,82.346697
6742,12/1/2024,12/31/2024,"Floyd County, IN",Indiana,Multi-Family (2-4 Unit),195000.0,252450.0,150.000000
7526,4/1/2017,4/30/2017,"Floyd County, IN",Indiana,Condo/Co-op,81900.0,179950.0,68.136439
...,...,...,...,...,...,...,...,...
1040934,9/1/2020,9/30/2020,"Floyd County, IN",Indiana,All Residential,200250.0,199900.0,112.685115
1041413,4/1/2024,4/30/2024,"Floyd County, IN",Indiana,All Residential,300000.0,279900.0,150.660478
1042398,12/1/2020,12/31/2020,"Floyd County, IN",Indiana,All Residential,203000.0,185000.0,113.318379
1045086,5/1/2015,5/31/2015,"Floyd County, IN",Indiana,All Residential,163000.0,149900.0,82.408342


,period_begin,period_end,region,state,property_type,median_sale_price,median_list_price,median_ppsf
244,2/1/2024,2/29/2024,"Bartholomew County, IN",Indiana,Condo/Co-op,202500.0,278200.0,156.976744
874,10/1/2012,10/31/2012,"Bartholomew County, IN",Indiana,Multi-Family (2-4 Unit),128500.0,139900.0,262.281473
3643,6/1/2014,6/30/2014,"Bartholomew County, IN",Indiana,Multi-Family (2-4 Unit),133900.0,129900.0,NaN
5908,10/1/2013,10/31/2013,"Bartholomew County, IN",Indiana,Condo/Co-op,77500.0,129900.0,62.299035
7783,4/1/2020,4/30/2020,"Bartholomew County, IN",Indiana,All Residential,167900.0,208950.0,91.174325
...,...,...,...,...,...,...,...,...
1037457,5/1/2015,5/31/2015,"Bartholomew County, IN",Indiana,Condo/Co-op,108750.0,146200.0,109.516048
1038601,6/1/2024,6/30/2024,"Bartholomew County, IN",Indiana,Single Family Residential,278000.0,264950.0,142.857143
1040444,12/1/2012,12/31/2012,"Bartholomew County, IN",Indiana,Single Family Residential,154500.0,137450.0,76.554171
1043312,11/1/2014,11/30/2014,"Bartholomew County, IN",Indiana,All Residential,151250.0,145000.0,75.757576


I now have the housing data in three separate dataframes. I'll merge these into one. An outer join should be sufficient since there won't be anything they have in common besides Indiana. 

In [24]:
#join the Three cleaned datasets together
bart_clark = housing_data_bart.merge(housing_data_clark, how = 'outer')
bart_clark

bart_clark_floyd = bart_clark.merge(housing_data_floyd, how = 'outer')
bart_clark_floyd

#drop any rows with NaN values
bart_clark_floyd = bart_clark_floyd.dropna()

# Remove State Column - that information is included in the Region column
bart_clark_floyd = bart_clark_floyd.drop(columns=['state'])

#make a copy of the data
bart_clark_floyd.copy()

,period_begin,period_end,region,property_type,median_sale_price,median_list_price,median_ppsf
0,1/1/2012,1/31/2012,"Bartholomew County, IN",Multi-Family (2-4 Unit),64250.0,142900.0,22.801584
1,1/1/2012,1/31/2012,"Bartholomew County, IN",Single Family Residential,125000.0,142900.0,59.311224
2,1/1/2012,1/31/2012,"Clark County, IN",All Residential,124250.0,141400.0,66.903633
3,1/1/2012,1/31/2012,"Clark County, IN",Single Family Residential,121750.0,141400.0,65.195190
4,1/1/2012,1/31/2012,"Floyd County, IN",All Residential,140000.0,114950.0,69.771352
...,...,...,...,...,...,...,...
1313,9/1/2024,9/30/2024,"Clark County, IN",Single Family Residential,249900.0,260000.0,164.021164
1314,9/1/2024,9/30/2024,"Clark County, IN",Townhouse,219900.0,260000.0,167.606707
1315,9/1/2024,9/30/2024,"Floyd County, IN",All Residential,231250.0,257450.0,148.266423
1316,9/1/2024,9/30/2024,"Floyd County, IN",Condo/Co-op,155900.0,257450.0,134.355275


Now I'll remove any property type that isn't "Single Family Residential."

In [25]:
#clean the merged data by narrowing down property types
cleaned_bcf = bart_clark_floyd.loc[bart_clark_floyd["property_type"] == "Single Family Residential"]

cleaned_bcf

,period_begin,period_end,region,property_type,median_sale_price,median_list_price,median_ppsf
1,1/1/2012,1/31/2012,"Bartholomew County, IN",Single Family Residential,125000.0,142900.0,59.311224
3,1/1/2012,1/31/2012,"Clark County, IN",Single Family Residential,121750.0,141400.0,65.195190
5,1/1/2012,1/31/2012,"Floyd County, IN",Single Family Residential,140000.0,114950.0,69.771352
7,1/1/2013,1/31/2013,"Bartholomew County, IN",Single Family Residential,110000.0,110950.0,59.248555
10,1/1/2013,1/31/2013,"Clark County, IN",Single Family Residential,116000.0,120000.0,76.371585
...,...,...,...,...,...,...,...
1305,9/1/2023,9/30/2023,"Clark County, IN",Single Family Residential,246900.0,250000.0,156.583037
1307,9/1/2023,9/30/2023,"Floyd County, IN",Single Family Residential,289000.0,279950.0,151.166514
1310,9/1/2024,9/30/2024,"Bartholomew County, IN",Single Family Residential,279736.0,279736.0,138.020833
1313,9/1/2024,9/30/2024,"Clark County, IN",Single Family Residential,249900.0,260000.0,164.021164


Now I know I have all the data I want so I'm going to clean up the column headers.

In [26]:
"""
Cleans and processes the Bartholomew, Clark, and Floyd County Housing DataFrame.

    This function performs the following operations on the input DataFrame:
    1. Cleans column names (capitalizes and removes underscores).

    Parameters:
        bart_clark_floyd (pandas.DataFrame): The input housing dataset.

    Returns:
        pandas.DataFrame: The cleaned and processed DataFrame.

    Example:
        >>> clean_kick_starter = clean_ks(bart_clark_floyd)\"
"""
# function       
def clean_hd (cleaned_bcf) -> pd.DataFrame:

    # Clean column names
    cleaned_bcf.columns =  cleaned_bcf.columns.str.title()
    cleaned_bcf.columns =  cleaned_bcf.columns.str.strip().str.replace('_', ' ')

    return cleaned_bcf

final_bcf = clean_hd(cleaned_bcf)

display(final_bcf)


,Period Begin,Period End,Region,Property Type,Median Sale Price,Median List Price,Median Ppsf
1,1/1/2012,1/31/2012,"Bartholomew County, IN",Single Family Residential,125000.0,142900.0,59.311224
3,1/1/2012,1/31/2012,"Clark County, IN",Single Family Residential,121750.0,141400.0,65.195190
5,1/1/2012,1/31/2012,"Floyd County, IN",Single Family Residential,140000.0,114950.0,69.771352
7,1/1/2013,1/31/2013,"Bartholomew County, IN",Single Family Residential,110000.0,110950.0,59.248555
10,1/1/2013,1/31/2013,"Clark County, IN",Single Family Residential,116000.0,120000.0,76.371585
...,...,...,...,...,...,...,...
1305,9/1/2023,9/30/2023,"Clark County, IN",Single Family Residential,246900.0,250000.0,156.583037
1307,9/1/2023,9/30/2023,"Floyd County, IN",Single Family Residential,289000.0,279950.0,151.166514
1310,9/1/2024,9/30/2024,"Bartholomew County, IN",Single Family Residential,279736.0,279736.0,138.020833
1313,9/1/2024,9/30/2024,"Clark County, IN",Single Family Residential,249900.0,260000.0,164.021164


In [28]:
#checking datatypes before I move on 
final_bcf.dtypes

Period Begin          object
Period End            object
Region                object
Property Type         object
Median Sale Price    float64
Median List Price    float64
Median Ppsf          float64
dtype: object

There's sales data going all the way back to 2012. That will NOT be an accurate snapshot for current housing prices (thanks, COVID). Let's pull the last rolling 12 months. Since Period Begin and Period End are objects, I will need to change them to datetime datatypes so the computer will recognize them as timestamps. 

In [ ]:
#making a copy to avoid the SettingwithCopyWarning
final_bcfcopy = final_bcf.copy()

#it's an object, not datetime
#change to datetime
final_bcfcopy['Period Begin',] = pd.to_datetime(final_bcfcopy['Period Begin'], format='%m/%d/%Y')

#adjust Period End to datetime 
final_bcfcopy['Period End'] = pd.to_datetime(final_bcfcopy['Period End'], format='%m/%d/%Y')


# Filter for the last rolling 13 months (anything after February 1, 2024)
filtered_bcf = final_bcfcopy.loc[(final_bcfcopy['Period Begin'] >= '2024-02-01')]   

filtered_bcf

,Period Begin,Period End,Region,Property Type,Median Sale Price,Median List Price,Median Ppsf,"(Period Begin,)"
102,2025-01-01,2025-01-31,"Bartholomew County, IN",Single Family Residential,231250.0,299900.0,136.032850,2025-01-01
105,2025-01-01,2025-01-31,"Clark County, IN",Single Family Residential,250000.0,274900.0,145.509943,2025-01-01
107,2025-01-01,2025-01-31,"Floyd County, IN",Single Family Residential,253900.0,290000.0,137.500000,2025-01-01
211,2024-10-01,2024-10-31,"Bartholomew County, IN",Single Family Residential,277500.0,245000.0,133.991798,2024-10-01
218,2024-10-01,2024-10-31,"Floyd County, IN",Single Family Residential,235000.0,274900.0,155.259823,2024-10-01
322,2024-11-01,2024-11-30,"Bartholomew County, IN",Single Family Residential,274291.0,239900.0,145.772595,2024-11-01
326,2024-11-01,2024-11-30,"Clark County, IN",Single Family Residential,260000.0,244900.0,171.232980,2024-11-01
329,2024-11-01,2024-11-30,"Floyd County, IN",Single Family Residential,286500.0,269000.0,154.476351,2024-11-01
434,2024-12-01,2024-12-31,"Bartholomew County, IN",Single Family Residential,253000.0,237000.0,148.601399,2024-12-01
438,2024-12-01,2024-12-31,"Clark County, IN",Single Family Residential,234945.0,262990.0,155.327776,2024-12-01


Time to visualize this data. Yippee!

In [36]:
# Create a bar graph using plotly
fig = px.bar(filtered_bcf, x='Period Begin', y='Median Sale Price', color='Region', barmode='group',
             title='Median Sale Prices for the last rolling 12 months',
             labels={'Median Sale Price': 'Median Home Price ($)', 'Date': 'Period Begin'},
             height=400)

fig.show()

Rinse and repeat with Housing PPSF. 